# Setup Verification

Run every cell in order (Runtime → Run all, or Shift+Enter through each).

Each cell prints **PASS** or **FAIL** with a plain explanation.
All five must pass before you start phase 01.

> If a cell fails, check `TROUBLESHOOTING.md` for the matching error.
> Fix it, then re-run that cell.

## Check 1 — GPU connected

In [ ]:
import torch

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    mem  = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
    print(f'PASS  GPU: {name}  ({mem} GB)')
else:
    print('FAIL  No GPU found.')
    print('      Go to Runtime > Change runtime type > select T4 GPU, then re-run.')

## Check 2 — Repository cloned and structure looks right

In [ ]:
import os

# Expected directories relative to wherever this notebook lives
expected_dirs = [
    '../01_tokenization',
    '../02_embeddings_positional',
    '../03_attention',
    '../04_transformer_block',
    '../05_data_pipeline',
    '../06_training_loop',
    '../07_sampling_generation',
    '../08_deployment_hf',
]

expected_files = [
    '../model_config.py',
    '../gpt.py',
    '../train.py',
    '../generate.py',
]

missing_dirs  = [d for d in expected_dirs  if not os.path.isdir(d)]
missing_files = [f for f in expected_files if not os.path.isfile(f)]

if not missing_dirs and not missing_files:
    print('PASS  All 8 phase folders and 4 scaffold files found.')
else:
    print('FAIL  Some expected paths are missing:')
    for p in missing_dirs + missing_files:
        print(f'      missing: {p}')
    print()
    print('  This usually means one of two things:')
    print('  1. You forgot to %cd into your cloned repo folder.')
    print('     Run:  %cd llm-from-scratch  (or whatever your folder is named)')
    print('     Then re-run this cell.')
    print('  2. You cloned the wrong URL. Check: !git remote -v')
    print('     The URL should contain YOUR username, not the instructor\'s.')

## Check 3 — Git identity configured

In [ ]:
import subprocess

def git_config(key):
    result = subprocess.run(
        ['git', 'config', key],
        capture_output=True, text=True
    )
    return result.stdout.strip()

name  = git_config('user.name')
email = git_config('user.email')

ok = bool(name) and bool(email)

if ok:
    print(f'PASS  Git identity: {name} <{email}>')
else:
    print('FAIL  Git identity not configured.')
    if not name:
        print('      user.name is not set.')
    if not email:
        print('      user.email is not set.')
    print()
    print('  Fix: run the following two shell commands in a code cell:')
    print('      !git config --global user.name  "Your Name"')
    print('      !git config --global user.email "your@email.com"')
    print('  Then re-run this cell.')

## Check 4 — Can commit and push to your fork

> This cell creates a temporary file, commits it, pushes it to your fork,
> then immediately deletes it and pushes again. It leaves no permanent trace.
> If your PAT authentication is not set up, this will fail.

In [ ]:
import subprocess, os

def run(cmd, **kwargs):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True, **kwargs)

# Write a temporary marker file
test_file = '__setup_verify_test__.txt'
with open(test_file, 'w') as f:
    f.write('setup verification test — safe to delete\n')

steps = [
    ('git add ' + test_file,           'Stage test file'),
    ('git commit -m "setup verify"',   'Commit test file'),
    ('git push',                        'Push to fork'),
]

failed = False
for cmd, label in steps:
    result = run(cmd)
    if result.returncode != 0:
        print(f'FAIL  {label} failed.')
        print(f'      Command: {cmd}')
        print(f'      Error output:')
        for line in (result.stderr or result.stdout).strip().split('\n'):
            print(f'        {line}')
        print()
        print('  Check TROUBLESHOOTING.md — section 2 (auth) or section 3 (wrong URL).')
        failed = True
        break

if not failed:
    # Clean up: remove test file and push the deletion
    os.remove(test_file)
    run('git add ' + test_file)
    run('git commit -m "remove setup verify test file"')
    run('git push')
    print('PASS  Committed and pushed to your fork successfully.')
    print('      (Test file created and removed. Check your GitHub repo to confirm.)')

## Check 5 — Hugging Face token is accessible

Paste your HF token in the cell below where it says `HF_TOKEN = "hf_..."`. This cell does not push anything — it just checks that the token is valid and has write scope.

> Do not commit this cell if your token is pasted into it. Clear the token value before committing, or store it in Colab Secrets (Runtime > Manage secrets).

In [ ]:
# Paste your Hugging Face token here (starts with hf_)
HF_TOKEN = "hf_PASTE_YOUR_TOKEN_HERE"

if HF_TOKEN == "hf_PASTE_YOUR_TOKEN_HERE" or not HF_TOKEN.startswith("hf_"):
    print('SKIP  No token provided.')
    print('      Paste your HF token above (hf_...) and re-run this cell.')
    print('      You can generate one at: huggingface.co > Settings > Access Tokens')
    print('      Required scope: Write')
else:
    try:
        from huggingface_hub import HfApi
        api  = HfApi(token=HF_TOKEN)
        info = api.whoami()
        # Check token has write permission by inspecting its role
        token_info = api.get_token_permission(HF_TOKEN)
        has_write  = getattr(token_info, 'access', None) in ('write', 'admin', None)
        if has_write:
            print(f'PASS  HF token valid. Logged in as: {info["name"]}')
            print('      Token has write access — ready for phase 08.')
        else:
            print(f'WARN  HF token valid. Logged in as: {info["name"]}')
            print('      But the token may be read-only. Phase 08 requires write scope.')
            print('      If phase 08 push fails, regenerate the token with Write role.')
    except ImportError:
        print('INFO  huggingface_hub not installed in this session.')
        print('      Run: !pip install huggingface_hub  then re-run this cell.')
    except Exception as e:
        print('FAIL  HF token is invalid or expired.')
        print(f'      Error: {e}')
        print('      Check TROUBLESHOOTING.md — section 5 (HF 403).')

## Summary

All 5 checks passed? You're ready to start **01_tokenization**.

| Check | What it verified |
|---|---|
| 1 | GPU runtime is connected |
| 2 | Repo is cloned and all phase folders + scaffold files are present |
| 3 | Git identity (name + email) is configured for this session |
| 4 | Can commit and push to your fork (authentication works) |
| 5 | Hugging Face token is valid and accessible |

> **Remember:** Checks 3 and 4 reset every Colab session. Run them again at the start of any session where you plan to commit work.